# Vertex AI 最小構成チュートリアル: カリフォルニア住宅価格推論

Notebook 1 つで Vertex AI の流れを体験する。パイプライン・コンテナビルド不要。

```
① ローカルで学習 → GCS にモデル保存
② Model Registry に登録
③ エンドポイントにデプロイ → 推論
```

所要時間: 約 15〜20 分（デプロイ待ち含む）

## 前提

```bash
gcloud services enable aiplatform.googleapis.com --project=mlops-dev-a
```

In [1]:
!pip install google-cloud-aiplatform "scikit-learn>=1.3,<1.4" pandas

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 17.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 22.6 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:━━━━━━━━━━━━━━━━━━━ 0/2 [numpy]
      Successfully uninstalled numpy-2.2.6━━━━━━ 0/2 [numpy]
  Attempting uninstall: scikit-learn━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: scikit-learn 1.7.232m0/2 [numpy]
    Uninstalling scikit-learn-1.7.2:m╺━━━━━━━━━━━━━━━━━━━ 1/2 [scikit-learn]
      Successfully uninstalled scikit-learn-1.7.2━━━━━━━━━━━━━ 1/2 [scikit-learn]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-learn] [scikit-learn]


## ① 学習 & GCS にモデル保存

In [2]:
import pickle
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from google.cloud import storage

# --- データ取得 ---
housing = fetch_california_housing(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42
)

# --- 学習 ---
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# --- 評価 ---
y_pred = model.predict(X_test)
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")

/home/ubuntu/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


RMSE: 0.5445
MAE:  0.3663


In [3]:
BUCKET = "mlops-dev-a-vertex"
MODEL_PATH = "models/california-housing-mvp/model.pkl"

with open("/tmp/model.pkl", "wb") as f:
    pickle.dump(model, f)

client = storage.Client(project="mlops-dev-a")
bucket = client.bucket(BUCKET)
bucket.blob(MODEL_PATH).upload_from_filename("/tmp/model.pkl")
print(f"保存完了: gs://{BUCKET}/{MODEL_PATH}")

保存完了: gs://mlops-dev-a-vertex/models/california-housing-mvp/model.pkl


> GCS バケットが未作成の場合:
> ```bash
> gcloud storage buckets create gs://mlops-dev-a-vertex --location=asia-northeast1
> ```

## ② Model Registry に登録

In [4]:
from google.cloud import aiplatform

aiplatform.init(project="mlops-dev-a", location="asia-northeast1")

model = aiplatform.Model.upload(
    display_name="california-housing-rf-mvp",
    artifact_uri=f"gs://{BUCKET}/models/california-housing-mvp/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
)

print(f"登録完了: {model.resource_name}")

/home/ubuntu/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/home/ubuntu/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/home/ubuntu/.local/lib/python3.10/site-packages/google/ap

Creating Model
Create Model backing LRO: projects/941178142366/locations/asia-northeast1/models/2907772848862920704/operations/4805007925259010048
Model created. Resource name: projects/941178142366/locations/asia-northeast1/models/2907772848862920704@1
To use this Model in another session:
model = aiplatform.Model('projects/941178142366/locations/asia-northeast1/models/2907772848862920704@1')
登録完了: projects/941178142366/locations/asia-northeast1/models/2907772848862920704


## ③ デプロイ & 推論

### デプロイ（5〜10 分待ち）

In [5]:
endpoint = model.deploy(
    deployed_model_display_name="california-housing-rf-mvp-v1",
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1,
)

print(f"エンドポイント: {endpoint.resource_name}")

Creating Endpoint
Create Endpoint backing LRO: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048/operations/550232177300733952
Endpoint created. Resource name: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048')
Deploying model to Endpoint : projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048
Deploy Endpoint model backing LRO: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048/operations/6204501509464391680
Endpoint model deployed. Resource name: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048
エンドポイント: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048


### 推論

In [6]:
# 8 特徴量: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
result = endpoint.predict(instances=[
    [8.3252, 41.0, 6.984, 1.024, 322.0, 2.556, 37.88, -122.23],
])

print(f"予測価格: ${result.predictions[0] * 100_000:,.0f}")

予測価格: $437,111


### テストデータで検証

In [7]:
instances = X_test.head(3).values.tolist()
result = endpoint.predict(instances=instances)

for i, (pred, actual) in enumerate(zip(result.predictions, y_test.head(3))):
    print(f"サンプル{i+1}: 予測=${pred * 100_000:,.0f}  実際=${actual * 100_000:,.0f}")

サンプル1: 予測=$56,803  実際=$47,700
サンプル2: 予測=$81,136  実際=$45,800
サンプル3: 予測=$483,575  実際=$500,001


## クリーンアップ（重要）

エンドポイントは常時課金されるため、確認が終わったら必ず削除する。

In [8]:
endpoint.undeploy_all()
endpoint.delete()
model.delete()
print("クリーンアップ完了")

Undeploying Endpoint model: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048
Undeploy Endpoint model backing LRO: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048/operations/9091308870608879616
Endpoint model undeployed. Resource name: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048
Deleting Endpoint : projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048
Endpoint deleted. . Resource name: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048
Deleting Endpoint resource: projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048
Delete Endpoint backing LRO: projects/941178142366/locations/asia-northeast1/operations/5781163144491565056
Endpoint resource projects/941178142366/locations/asia-northeast1/endpoints/6150775797918466048 deleted.
Deleting Model : projects/941178142366/locations/asia-northeast1/models/2907772848862920704
Model 

In [9]:
# GCS のモデルも削除する場合
!gcloud storage rm -r gs://mlops-dev-a-vertex/models/california-housing-mvp/

Removing objects:
Removing gs://mlops-dev-a-vertex/models/california-housing-mvp/model.pkl#1774514146705300...
  Completed 1/1                                                                
